# Verify Conv-VAE-Neo MultiView Concat

In [ ]:
import json
import pathlib
import sys
sys.path.append("..")

import matplotlib.pyplot as plt
import torch
from exp_run_config import Config
Config.PROJECTNAME = "BerryPicker"
from sensorprocessing.conv_vae_neo import ConvVAENeoLoss
from sensorprocessing.conv_vae_neo_multiview_concat import make_dataloaders
from sensorprocessing.sp_factory import create_sp

In [ ]:
experiment = "sensorprocessing_conv_vae_neo_multiview_concat"
run = "sp_vae_neo_multiview_concat_128_256px"
creation_style = "exist-ok"

In [ ]:
exp = Config().get_experiment(experiment, run, creation_style=creation_style)
sp = create_sp(exp)
_, validation_loader = make_dataloaders(exp)
views = next(iter(validation_loader))
views = [view.to(Config().runtime["device"]) for view in views]
with torch.no_grad():
    output = sp.enc(views)
composite = sp.enc.compose_views(views)
components = ConvVAENeoLoss(exp).components(output, composite)
per_camera_mse = [
    float(torch.mean((original - reconstructed) ** 2))
    for original, reconstructed in zip(
        sp.enc.split_composite(composite), sp.enc.split_composite(output[0])
    )
]
metrics = {
    "loss": float(components["loss"]),
    "reconstruction_loss": float(components["reconstruction"]),
    "kl_loss": float(components["kl"]),
    "per_camera_mse": dict(zip(sp.cameras, per_camera_mse)),
    "latent_shape": list(output[1].shape),
}
print(json.dumps(metrics, indent=2))
metrics_path = pathlib.Path(exp["data_dir"], "verification_metrics.json")
with metrics_path.open("w", encoding="utf-8") as handle:
    json.dump(metrics, handle, indent=2)
    handle.write("\n")

In [ ]:
original_views = sp.enc.split_composite(composite.cpu())
reconstructed_views = sp.enc.split_composite(output[0].cpu())
count = min(4, composite.size(0))
fig, axes = plt.subplots(2 * sp.num_views, count, figsize=(3 * count, 6 * sp.num_views), squeeze=False)
for view_index, camera in enumerate(sp.cameras):
    for sample_index in range(count):
        axes[2 * view_index, sample_index].imshow(original_views[view_index][sample_index].permute(1, 2, 0))
        axes[2 * view_index, sample_index].axis("off")
        axes[2 * view_index + 1, sample_index].imshow(reconstructed_views[view_index][sample_index].permute(1, 2, 0))
        axes[2 * view_index + 1, sample_index].axis("off")
    axes[2 * view_index, 0].set_title(f"{camera} original")
    axes[2 * view_index + 1, 0].set_title(f"{camera} reconstruction")
fig.tight_layout()
figure_path = pathlib.Path(exp["data_dir"], "verification_reconstructions.png")
fig.savefig(figure_path, bbox_inches="tight")
plt.show()

In [ ]:
with torch.no_grad():
    samples = sp.enc.sample(4).cpu()
sample_views = sp.enc.split_composite(samples)
fig, axes = plt.subplots(sp.num_views, 4, figsize=(12, 3 * sp.num_views), squeeze=False)
for view_index, camera in enumerate(sp.cameras):
    for sample_index in range(4):
        axes[view_index, sample_index].imshow(sample_views[view_index][sample_index].permute(1, 2, 0))
        axes[view_index, sample_index].axis("off")
    axes[view_index, 0].set_title(f"{camera} prior sample")
fig.tight_layout()
sample_path = pathlib.Path(exp["data_dir"], "verification_prior_samples.png")
fig.savefig(sample_path, bbox_inches="tight")
plt.show()